# Tracing LLM Apps: Langfuse & OpenTelemetry

Wiki reference for [Langfuse & OpenTelemetry](https://ml-viz-ruby.vercel.app/wiki/langfuse-and-opentelemetry).

**The idea in one sentence.** Observability for LLM apps is built on **distributed tracing**: a
request becomes a tree of **spans** (retrieval, embedding, generation), each tagged with
standard `gen_ai.*` attributes (model, token counts) — so you can attribute **latency and cost**
to the exact step that caused them.

We implement a minimal tracer and cost model from scratch, **validate the trace tree and its
cost attribution**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
from dataclasses import dataclass, field
from contextlib import contextmanager

import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

## 1 — A minimal tracer

A **span** is a timed operation with a name, attributes, and a parent — parents make the tree.
A **tracer** hands out spans and tracks the current parent via a stack. To keep runs
deterministic we use a simulated clock: instead of `time.time()`, instrumented code calls
`advance(ms)` to model work taking time.

In [ ]:
@dataclass
class Span:
    name: str
    span_id: int
    parent_id: int | None
    start: float                    # ms since trace start
    end: float = 0.0
    attributes: dict = field(default_factory=dict)

class Tracer:
    def __init__(self):
        self.spans, self.clock, self._stack, self._next_id = [], 0.0, [], 1

    def advance(self, ms):          # simulated work
        self.clock += ms

    @contextmanager
    def start_span(self, name, **attributes):
        span = Span(name, self._next_id,
                    self._stack[-1].span_id if self._stack else None,
                    start=self.clock, attributes=dict(attributes))
        self._next_id += 1
        self.spans.append(span)
        self._stack.append(span)
        try:
            yield span
        finally:
            span.end = self.clock
            self._stack.pop()

## 2 — Instrumenting a RAG request with GenAI conventions

The [GenAI semantic conventions](https://ml-viz-ruby.vercel.app/wiki/langfuse-and-opentelemetry)
standardize attribute names (`gen_ai.operation.name`, `gen_ai.usage.input_tokens`, …) so every
backend means the same thing by "input tokens". We instrument a mock RAG pipeline: input
guardrail → retrieval (with an embeddings call) → chat generation → output guardrail.

In [ ]:
PRICES = {  # $ per 1M tokens: (input, output)
    'claude-sonnet-5':    (3.00, 15.00),
    'text-embedding-3-s': (0.02,  0.00),
}

def llm_call(tracer, model, operation, in_tok, out_tok, latency_ms, **extra):
    with tracer.start_span(f"{operation} {model}", **{
        'gen_ai.operation.name': operation,
        'gen_ai.request.model': model,
        'gen_ai.usage.input_tokens': in_tok,
        'gen_ai.usage.output_tokens': out_tok,
        **extra,
    }):
        tracer.advance(latency_ms)

def rag_request(tracer, question):
    with tracer.start_span('rag-answer', **{'user.id': 'u_812', 'session.id': 's_33'}):
        with tracer.start_span('guardrail:input', flagged=False):
            tracer.advance(12)
        with tracer.start_span('retrieval', k=8, index='docs-v3'):
            llm_call(tracer, 'text-embedding-3-s', 'embeddings', 41, 0, 95)
            tracer.advance(85)                       # ANN search
        llm_call(tracer, 'claude-sonnet-5', 'chat', 1742, 319, 1650,
                 **{'gen_ai.request.temperature': 0.2, 'langfuse.prompt': 'rag-answer v14'})
        with tracer.start_span('guardrail:output', flagged=False):
            tracer.advance(3)

tracer = Tracer()
rag_request(tracer, 'What is our refund policy?')
print(f"{len(tracer.spans)} spans, trace duration = {max(s.end for s in tracer.spans):.0f} ms")

## 3 — The Langfuse-style layer: cost and scores

A generic APM stops at timed intervals. An LLM-native backend knows that spans with
`gen_ai.usage.*` attributes are **generations** and computes dollars from a price table,
and it lets you attach **scores** (user feedback, judge grades) to the trace — turning the
trace store into a filterable evaluation dataset.

In [ ]:
def span_cost(span, prices=PRICES):
    a = span.attributes
    model = a.get('gen_ai.request.model')
    if model not in prices:
        return 0.0
    p_in, p_out = prices[model]
    return (a.get('gen_ai.usage.input_tokens', 0) * p_in
          + a.get('gen_ai.usage.output_tokens', 0) * p_out) / 1e6

def render(spans):
    children = {}
    for s in spans:
        children.setdefault(s.parent_id, []).append(s)
    def walk(pid, depth):
        for s in children.get(pid, []):
            cost = span_cost(s)
            tag = f"  ${cost:.4f}" if cost else ""
            print(f"{'  ' * depth}{s.name:<28} {s.end - s.start:7.0f} ms{tag}")
            walk(s.span_id, depth + 1)
    walk(None, 0)

render(tracer.spans)
scores = {'user-feedback': 0, 'judge-faithfulness': 0.4}   # both low → inspect this trace
print(f"\ntrace cost = ${sum(span_cost(s) for s in tracer.spans):.4f}   scores = {scores}")

### Validate: cost is attributed to the LLM/embedding spans

Each generation span carries `gen_ai.usage.*` token counts, so a per-span cost model turns the
trace into a **cost breakdown**. Non-LLM spans (retrieval, glue code) carry no token cost. We
confirm the trace has a positive total cost concentrated in the model spans.

In [ ]:
total = sum(span_cost(s) for s in tracer.spans)
gen = [s for s in tracer.spans if 'gen_ai.request.model' in s.attributes]
print(f'{len(tracer.spans)} spans, {len(gen)} LLM/embedding calls, total cost ${total:.5f}')
assert total > 0 and len(gen) >= 1, 'the trace attributes cost to the LLM/embedding spans'
assert all(span_cost(s) == 0 for s in tracer.spans if 'gen_ai.request.model' not in s.attributes), 'non-LLM spans carry no token cost'
print('\n✅ standard gen_ai.* attributes turn a trace into a per-step cost breakdown')

## 4 — The trace waterfall

The standard trace view: one horizontal bar per span, indented by depth, positioned by start
time. Overlap (none here — this trace is fully sequential) is exactly what reveals
parallelism opportunities.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
spans = tracer.spans
for i, s in enumerate(spans):
    is_gen = 'gen_ai.request.model' in s.attributes
    ax.barh(len(spans) - 1 - i, s.end - s.start, left=s.start, height=0.6,
            color='#6366f1' if is_gen else '#2dd4bf', alpha=0.9)
ax.set_yticks(range(len(spans)))
ax.set_yticklabels([s.name for s in reversed(spans)], fontsize=8)
ax.set_xlabel('ms since trace start')
ax.set_title('RAG request — trace waterfall (indigo = LLM generations)')
plt.tight_layout(); plt.show()

### Validate: the spans form a trace tree

A trace is a tree: one **root** span (the request) with child spans nested inside it. Every
non-root span points at a real parent — which is what lets the waterfall show *which step*
dominated latency. We confirm the tree structure.

In [ ]:
roots = [s for s in tracer.spans if s.parent_id is None]
ids = {s.span_id for s in tracer.spans}
print(f'root spans: {len(roots)}; every child references a parent: {all(s.parent_id in ids for s in tracer.spans if s.parent_id is not None)}')
assert len(roots) == 1, 'a trace has a single root span (the request)'
assert all(s.parent_id in ids for s in tracer.spans if s.parent_id is not None), 'child spans reference their parent -> a valid tree'
print('\n✅ the span tree is what powers the latency waterfall and cost rollups')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no tracing** | you see one cost/latency lump, not where it came from (demo) |
| **missing attributes** | can't attribute cost without token/model tags |
| **broken parent links** | the waterfall / rollups break (verified needs a tree) |
| **sampling** | tracing every request is costly at scale — sample |
| **PII in spans** | prompts/outputs in traces can leak sensitive data |

Demo: per-model cost breakdown reconciles with the trace total.

In [ ]:
# The payoff of tracing: you can slice cost by ANY attribute — here, by MODEL. Without spans you
# only see one lump sum; with them you see exactly where the money goes (which is often a
# surprise). We build a per-model cost breakdown and confirm it reconciles with the trace total.
def cost_by_model(spans):
    totals = {}
    for s in spans:
        m = s.attributes.get('gen_ai.request.model')
        if m:
            totals[m] = totals.get(m, 0.0) + span_cost(s)
    return totals
cbm = cost_by_model(tracer.spans)
for model, c in cbm.items():
    print(f'  {model:<22} ${c:.5f}')
assert abs(sum(cbm.values()) - sum(span_cost(s) for s in tracer.spans)) < 1e-9, 'per-model costs reconcile with the trace total'
print('\nTracing lets you attribute cost/latency to a model, a step, a user -> the basis of LLM observability.')

## ✏️ Your turn

**Concept recap.** Because every generation span carries the *same* standardized attributes,
fleet-wide questions become one aggregation. "What did each model cost us?" is a groupby over
`gen_ai.request.model` — no log parsing.

**Exercise.** Implement `cost_by_model(spans, prices)`: return a dict mapping each model that
appears in the spans to its **total** dollar cost, computed from `gen_ai.usage.input_tokens` /
`gen_ai.usage.output_tokens` and the `(input, output)` per-1M-token prices. Spans without a
model attribute contribute nothing.

In [ ]:
def cost_by_model(spans, prices=PRICES):
    totals = {}
    for span in spans:
        # TODO(you): read the model from span.attributes; skip spans without one;
        #            accumulate (in_tok * p_in + out_tok * p_out) / 1e6 into totals[model]
        ...
    return totals

cost_by_model(tracer.spans)

In [ ]:
# This assert cell passes silently when your implementation is correct.
big = Tracer()
for _ in range(3):
    rag_request(big, 'q')
got = cost_by_model(big.spans)
assert set(got) == {'claude-sonnet-5', 'text-embedding-3-s'}, f"models: {sorted(got)}"
assert abs(got['claude-sonnet-5'] - 3 * (1742 * 3.00 + 319 * 15.00) / 1e6) < 1e-9
assert abs(got['text-embedding-3-s'] - 3 * (41 * 0.02) / 1e6) < 1e-12
assert abs(sum(got.values()) - sum(span_cost(s) for s in big.spans)) < 1e-9
print('✓ cost_by_model matches per-span costs across the fleet')

<details>
<summary>Solution</summary>

```python
def cost_by_model(spans, prices=PRICES):
    totals = {}
    for span in spans:
        a = span.attributes
        model = a.get('gen_ai.request.model')
        if model is None or model not in prices:
            continue
        p_in, p_out = prices[model]
        cost = (a.get('gen_ai.usage.input_tokens', 0) * p_in
              + a.get('gen_ai.usage.output_tokens', 0) * p_out) / 1e6
        totals[model] = totals.get(model, 0.0) + cost
    return totals
```

The whole exercise is one dictionary fold — which is the point: standardized attribute names
turn cost accounting into trivial aggregation.
</details>

## Key takeaways

- **A trace is a tree of spans** (retrieval → embed → generate) with one root (verified).
- **Standard `gen_ai.*` attributes** carry model + token usage, enabling a cost model (verified).
- **Attribute cost/latency to the exact step** — or model, user, etc. (demo).
- **OpenTelemetry semantics** make traces portable across tools (Langfuse, etc.).